# **05 - SQLite3 + pathlib: Crear BD y CRUD**



In [20]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path # Importante para ver las rutas de la base de datos

# 5.1 Librería pathlib: ruta y creación de la base de datos
Con la **librería pathlib** primero se van a verificar las rutas hacia la base de datos 

### 5.1.1 Métodos de la librería pathlib para comprobar las RUTAS:
* `Path(ruta/archivo.db)**.resolve()**` # devuelve la ruta absoluta
* `Path(ruta/archivo.db)`sin *resolve()* # devuelve la ruta relativa

### 5.1.2 Propiedades:
* Path(ruta/archivo.db)**.parent** # devuelve el directorio

### 5.1.3 Métodos:
* **exists()** # Devuelve un booleano si existe o no el directorio padre
* **mkdir()** # crea el directorio que falta 
    - ejemplo: `nueva_base_de_datos.parent.mkdir(parents=True) # Se creo el directorio data`
    - Atención: *parents* es el parámetro de mkdir() **parent** *en singular* es una PROPIEDAD
    - parents=False no crea el directorio y lanza FileNotFoundError
* resolve() visto arriba

### 5.1.4 Hay que ver algo de la librería sqlite3 ahora: creación de una base de datos con squlite.connect()
El método de creación base de datos.
método **.connect(ruta/archivo)**

Se guarda en una variable que se utilizará después
```python
conexion = squlite3.connect(ruta/archivo.db)
```

- Si el archivo no existe, lo crea.
- Si la ruta no existe lanza la excepción `sqlite3.OperationalError`
- Si no se tienen permisos de escritura lanza la excepción `sqlite3.OperationalError`

In [21]:
# CÓDIGO DE LA SECCIÓN 5.1 Librería pathlib: ruta y creación de la base de datos

# 5.1.1 MÉTODO .RESOLVE() - Path().resolve()
nueva_base_de_datos = Path("nueva_base_de_datos.db") # Ejemplo: No se le indica ningún subdirectorio
print(nueva_base_de_datos.resolve()) # Esto muestra la ruta completa 

nueva_base_de_datos = Path("data/nueva_base_de_datos.db")
print(nueva_base_de_datos.resolve()) # Ejemplo: Esto muestra la ruta completa con el subdirectorio (aunque no exista)
# por sí solo estos métodos no lanzan error si no existe el directorio

# 5.1.2 PROPIEDAD .PARENT y MÉTODO .EXIST() - Path().parent.exists()
# Comprobación de existencia del directorio padre 'data/'
# .parent es una propiedad de Path que devuelve el directorio padre
# .exists() es un método que devuelve True si la ruta existe, False si no existe
if not nueva_base_de_datos.parent.exists():
        print("Mensaje desde dentro del IF:")
        print(f"Error: La carpeta '{nueva_base_de_datos.parent}' no existe.")
        print(f'Booleano que devuelve el método exists(): {nueva_base_de_datos.parent.exists()}')

# 5.1.3 EXCEPCIÓN Y MÉTODO MKDIR() - la excepción se lanza al intentar crear la base de datos en un directorio que no existe
# Como en el ejemplo no existe el directorio 'data' se crea tras capturar la excepción.

try: # La ruta sigue siendo Path("data/nueva_base_de_datos.db")
    conexion = sqlite3.connect(nueva_base_de_datos) # Crearía la base de datos si no existiera ya en el directorio EXISTENTE
except sqlite3.OperationalError as e: # Captura la excepción si el directorio no existe
    print("Mensaje desde dentro de la EXCEPCIÓN:")
    print("No se ha podido crear la base de datos porque no existe el directorio 'data'.")
    print("Error:", e)
    # Este bloque crea el directorio y la base de datos
    nueva_base_de_datos.parent.mkdir(parents=True) # Se crea el directorio 'data'
    print(f"Se ha creado el directorio: {nueva_base_de_datos.parent}") # print para confirmar
    conexion = sqlite3.connect(nueva_base_de_datos) # Se crea la base de datos Ruta y archivo data/nueva_base_de_datos.db
    print("Ahora se ha creado la base de datos correctamente después de crear el directorio.") # print para confirmar
    print("Fin del código de la excepción.")

# 5.1.4 MÉTODO DE SQLITE3 .CONNECT: Crear o conectar con la base de datos: OJO MÉTODO DE SQLITE3
conexion = sqlite3.connect(nueva_base_de_datos) # Crearía la base de datos si no existiera aún en el directorio EXISTENTE
# Como existe el directorio 'data/' no hay error
# Como existe la base de datos, no se crea una nueva sino que se conecta a la existente
print("Ejecución normal: Conexión a la base de datos establecida correctamente.") # print para confirmar


C:\Users\angel\source\CEPython2526\Modulo_3_Analisis_de_datos\ejercicios\00_R_numpy_pandas\nueva_base_de_datos.db
C:\Users\angel\source\CEPython2526\Modulo_3_Analisis_de_datos\ejercicios\00_R_numpy_pandas\data\nueva_base_de_datos.db
Ejecución normal: Conexión a la base de datos establecida correctamente.


# 5.2 Creación de un DataFrame para la base de datos y su conversión de tipos de datos - Librería PANDAS
Se crea un **DataFrame** con **Series** para una **Tabla** de eventos históricos 'eventos':
- *id* se forma con un rango de Numpy
- *evento* en principio tiene el tipo object y se convierte a string
- *fecha* en principio tiene el tipo object y se convierte después a datetime64 con el día delante `dayfirst=True`
- *belico* un boleeano que indica si el evento fue un evento bélico.

SIN COMPLICACIONES: se crea el DataFrame con `df.Series()`

*Curiosidad: Pandas no ha aceptado ni la 'Caída de Constantinopla' el 29-05-1453 ni el 'Descubrimiento de América' el 12-10-1492 porque las fechas se tienen que encontrar entre el año 1677 y el año 2262.*

- 1 Revolución Francesa 05-05-1789 True
- 2 Inauguración de la Torre Eiffel 15-05-1889 False
- 3 Primera Guerra Mundial 28-07-1914 True
- 4 Bomba de Hiroshima 06-08-1945 True
- 5 Llegada a la Luna 16-07-1969 False
- 6 Primera transmisión de Arpanet 29-10-1969 False
- 7 Primera proyección de los hermanos Lumière 28-12-1895 False


In [22]:
# CÓDIGO DE LA SECCIÓN 5.2 Creación de un DataFrame para la base de datos y su conversión de tipos de datos - Librería PANDAS

# 5.2.1 Creación de las Series para el DataFrame
id = pd.Series(np.arange(1,8))

# 'Caída de Constantinopla','Descubrimiento de América','29-05-1453','12-12-1492',True, False,
evento = pd.Series(['Revolución Francesa',
                    'Inauguración de la Torre Eiffel','Primera Guerra Mundial','Bomba de Hiroshima',
                    'Llegada a la Luna','Primera transmisión de Arpanet','Primera proyección de los hermanos Lumière'])

fecha = pd.Series(['05-05-1789','15-05-1889','28-07-1914','06-08-1945',
                       '16-07-1969','29-10-1969','28-12-1895'])

belico = pd.Series([True, False, True, True, False, False, False]) # True si fue un evento bélico

# 5.2.2 Creación del DataFrame
df_eventos = pd.DataFrame({'id': id, 'evento': evento, 'fecha': fecha, 'belico': belico})

# 5.2.3 Comprobación y onversión de tipos de datos del DataFrame
display(df_eventos)
print("\nTipos de datos originales:")
print("evento y fecha son 'object' por defecto al crear el DataFrame:")
display(df_eventos.dtypes)

# Conversión de tipos de datos
print("\nTipos de datos después de la conversión:")
print("Conversión de evento a string y fecha a datetime:")
df_eventos["evento"] = df_eventos["evento"].astype("string") # Conversión de evento a string
df_eventos["fecha"] = pd.to_datetime(df_eventos["fecha"], dayfirst=True) # Conversión a datetime con día primero
display(df_eventos.dtypes)

,id,evento,fecha,belico
0,1,Revolución Francesa,05-05-1789,True
1,2,Inauguración de la Torre Eiffel,15-05-1889,False
2,3,Primera Guerra Mundial,28-07-1914,True
3,4,Bomba de Hiroshima,06-08-1945,True
4,5,Llegada a la Luna,16-07-1969,False
5,6,Primera transmisión de Arpanet,29-10-1969,False
6,7,Primera proyección de los hermanos Lumière,28-12-1895,False



Tipos de datos originales:
evento y fecha son 'object' por defecto al crear el DataFrame:


id         int64
evento    object
fecha     object
belico      bool
dtype: object


Tipos de datos después de la conversión:
Conversión de evento a string y fecha a datetime:


id                 int64
evento    string[python]
fecha     datetime64[ns]
belico              bool
dtype: object

# 5.3 Librería Pandas SOBRETODO - Creación de la TABLA, subida del DataFrame a la base de datos y comprobación

## 5.3.1 Ruta-Path
Obtener la ruta de la base de datos con SQLite3 para la conexión VER 5.1.1 `db_path = base_datos.resolve()`

## 5.3.2 - MÉTODO DE PANDAS DATAFRAME.copy()
Copia del DataFrame para subir a la base de datos
`df_a_subir = df_origina.copy()`

### Explicación ampliada:
- A **Sin .copy()**: Si la función que sube los datos realiza alguna modificación interna en el DataFrame (aunque no se modifique el DataFrame original en el código), puede afectar al DataFrame original.
- B **Con .copy()**: Se crea una copia independiente del DataFrame original, evitando cualquier modificación no deseada.
- **ASÍ QUE HAY QUE USAR .COPY() PARA EVITAR PROBLEMAS!!!**

## 5.3.3 - Se crea la conexión a la base de datos con SQLite3
VER 5.1.3 y 5.1.4
`conexion=sqlite3.connect(db_path)`

## 5.3.4 - OJO MÉTODO DE PANDAS DF-COPIADO.to_sql(MUCHOS PARÁMETROS)
**REPETIMOS**
`df_copiado.to_sql(MUUUUUCHOS PARÁMETROS)`

`df_copiado.to_sql("eventos", conexion, index=False, if_exists="replace")`

**EXPLICACIÓN DE LOS PARÁMETROS**

- **name**: en el ejemplo "*eventos*": **OBLIGATORIO** es el nombre de la tabla
- **con**: en el ejemplo "*conexion*": **OBLIGATORIO** es la CONEXIÓN A LA BASE DE DATOS
- **if_exists**: ¿qué hacer si ya existe la tabla? - Posibilidades:
    - **replace**: reemplazar la tabla
    - **append**: agregar las filas
    - **fail**: lanzar un error

- **dtype**: este parámetro es un **DICCIONARIO** que indica el tipo de datos EN SQL 
    **AMPLIACIÓN dtype**: SE PONEN TIPOS EN SQL 'VARCHAR' **OK** // 'STRING' **MAL**
    tipos_sql = {
    'id': types.Integer                
    'evento': types.VARCHAR(length=100)
}

*DOS PARÁMETROS CONFUSOS:*
- **index**: Booleano- True indica que **EL ÍNDICE DEL DATAFRAME** se escribirá como una columna en la tabla
- **index_label**: index_label solo se usa si index es True Y SE PONE EL NOMBRE `index_label = "mi_indice"`
- Posibilidades:
    - **index=False**	Solo se crean las columnas del DataFrame. El índice desaparece.
    - **index=True** (*sin label*)	Se crea una columna extra llamada "index" o el nombre que tenga en el DataFrame
    - **index=True, index_label = "mi_indice"**: Se crea la columna con nombre "*mi_indice*"
- **index_label NO SIRVE PARA RENOMBRAR LAS COLUMNAS**

## 5.3.5 - Comprobación de la subida correcta leyendo la tabla desde la base de datos
- Se utiliza el método de Pandas **read_sql_query()** con dos parámetros
- Parámetros:
    - Primero: la **consulta SQL**
    - Segundo: la **conexión** (*Ver 5.1.3 y 5.1.4 y 5.3.3*)
```
df_leido = pd.read_sql_query("SELECT * FROM eventos", conexion)
display(df_leido)
```
## 5.3.6 - Comprobación alternativa con try-except para capturar errores al subir el DataFrame
```
try:
    df_copiado.to_sql('eventos', conexion, if_exists='replace', index=False)
    print("¡DataFrame subido exitosamente!")
except Exception as e:
    print(f"Ocurrió un error al subir el DataFrame: {e}")
```

In [23]:
# CÓDIGO DE LA SECCIÓN 5.3 Librería Pandas SOBRETODO - Creación de la TABLA, subida del DataFrame a la base de datos y comprobación

# 5.3.1 - Obtener la ruta de la base de datos con SQLite3 para la conexión
db_path = nueva_base_de_datos.resolve() # Ver 5.1.1 - Se obtiene la ruta completa de la base de datos

# 5.3.2 - COPIA del DataFrame para la base de datos EVENTO DE PANDAS
# ESTE PASO ES NUEVO
df_copiado = df_eventos.copy() # Copia del DataFrame para subir a la base de datos

# Explicación ampliada:
# A) Sin .copy(): Si la función que sube los datos realiza alguna modificación interna en el DataFrame
# (aunque no se modifique el DataFrame original en el código), puede afectar al DataFrame original.
# B) Con .copy(): Se crea una copia independiente del DataFrame original, evitando cualquier modificación no deseada.
# ASÍ QUE HAY QUE USAR .COPY() PARA EVITAR PROBLEMAS!!!

# 5.3.3 - Se crea la conexión a la base de datos con SQLite3
conexion=sqlite3.connect(db_path)

# 5.3.4 - Subida del DataFrame a la base de datos MÉTODO to_sql() DE PANDAS
df_copiado.to_sql("eventos", conexion, index=False, if_exists="replace")

# 5.3.5 - Comprobación de la subida correcta leyendo la tabla desde la base de datos
df_leido = pd.read_sql_query("SELECT * FROM eventos", conexion)
print("\nDataFrame leído desde la base de datos SQLite3:")
display(df_leido)

# 5.3.6 - Comprobación alternativa con try-except para capturar errores al subir el DataFrame
try:
    df_copiado.to_sql('eventos', conexion, if_exists='replace', index=False)
    print("¡DataFrame subido exitosamente!")
except Exception as e:
    print(f"Ocurrió un error al subir el DataFrame: {e}")


DataFrame leído desde la base de datos SQLite3:


,id,evento,fecha,belico
0,1,Revolución Francesa,1789-05-05 00:00:00,1
1,2,Inauguración de la Torre Eiffel,1889-05-15 00:00:00,0
2,3,Primera Guerra Mundial,1914-07-28 00:00:00,1
3,4,Bomba de Hiroshima,1945-08-06 00:00:00,1
4,5,Llegada a la Luna,1969-07-16 00:00:00,0
5,6,Primera transmisión de Arpanet,1969-10-29 00:00:00,0
6,7,Primera proyección de los hermanos Lumière,1895-12-28 00:00:00,0


¡DataFrame subido exitosamente!


# 5.4 Métodos SQL cursor() y execute() - métodos para confirmar commit() y rollback()

## 5.4.1 Un **cursor** es el objeto que ejecuta SQL y recorre resultados.  
Se usa el método cursor() sobre conexión.
```
cursor = conexion.cursor()
conexion.execute(
    "INSERT INTO eventos (evento) VALUES (?)",
    ("Nochevieja de 1999",)
)
conexion.commit()
```
## 5.4.2 El método **execute()** se ejecuta sobre conexion. execute()  
execute() no permite
- reutilizar el cursor
- recorrer resultados fila a fila

```
conexion.execute(
    "INSERT INTO eventos (evento) VALUES (?)",
    ("Nochevieja de 1999",)
)
conexion.commit()
```

## 5.4.3 Parametrizar para evitar inyección SQL

Los valores se ponen aparte, de este modo se evita un string.  
*Esto de abajo no es seguro:*
```
conexion.execute(
    f"INSERT INTO eventos (evento) VALUES ('{evento_nuevo}')"
)
```
**ASÍ ES CÓMO DEBE HACERSE**
```
conexion.execute(
    "INSERT INTO eventos (evento) VALUES (?)",
    ("Nochevieja de 1999",)
)
conexion.commit()
```
**EL MISMO EJEMPLO DE OTRO MODO**
```
sql_insertar_nuevo_evento = """
    INSERT INTO eventos (id, evento) VALUES (?, ?)
"""
nuevo_id = 9
nuevo_evento = "Nochevieja de 1999"
conexion.execute(sql_insertar_nuevo_evento, (nuevo_id, nuevo_evento))
conexion.commit()
```

## 5.4.4 Métodos commit() y rollback()

Una vez se ha ejecutado el SQL hay que hacer una de estas dos operaciones: métodos sobre conexion
**Método commit()** para confirmar (si no, no se guardará en la base de datos).  
`conexion.commit()`

**Método rollback()** para retroceder y no se guardará en la base de datos).  
`conexion.rollback()`

In [ ]:
# CÓDIGO DE LA SECCIÓN 5.4 Métodos SQL cursor() y execute() - métodos para confirmar commit() y rollback()

# A) Inserción de un nuevo evento en la tabla usando SOLO execute
sql_insertar_nuevo_evento = """
    INSERT INTO eventos (id, evento) VALUES (?, ?)
"""
nuevo_id = 9
nuevo_evento = "Nochevieja de 1999"
conexion.execute(sql_insertar_nuevo_evento, (nuevo_id, nuevo_evento))

# B) Del punto 5.4.4 - Confirmación de la inserción con commit()
conexion.commit()
# Si se quiere deshacer la inserción, se puede usar rollback()
conexion.rollback() # Aquí no se deshace porque se ha confirmado con commit() antes

# C) Comprobación de la inserción leyendo de nuevo la tabla desde la base de datos
df_leido_actualizado = pd.read_sql_query("SELECT * FROM eventos", conexion)
print("\nDataFrame leído desde la base de datos SQLite3 después de insertar un nuevo evento:")
display(df_leido_actualizado)


DataFrame leído desde la base de datos SQLite3 después de insertar un nuevo evento:


,id,evento,fecha,belico
0,1,Revolución Francesa,1789-05-05 00:00:00,1.0
1,2,Inauguración de la Torre Eiffel,1889-05-15 00:00:00,0.0
2,3,Primera Guerra Mundial,1914-07-28 00:00:00,1.0
3,4,Bomba de Hiroshima,1945-08-06 00:00:00,1.0
4,5,Llegada a la Luna,1969-07-16 00:00:00,0.0
5,6,Primera transmisión de Arpanet,1969-10-29 00:00:00,0.0
6,7,Primera proyección de los hermanos Lumière,1895-12-28 00:00:00,0.0
7,9,Nochevieja de 1999,None,NaN
